<a href="https://colab.research.google.com/github/LorenzoBioinfo/import-biology/blob/main/notebooks/DEG_Analysis_pyDESeq2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Differential Gene Expression Analysis con pyDESeq2

In questo notebook, vedremo come eseguire una Differential gene expression (DEG) analysis.

Utilizzeremo il dataset : RNA-sequencing of pretreatment AB1 (mesothelioma) and Renca (kidney cancer) tumours mice both respondant and non-respondant to immunotherapy with CTLA4 and PD-L1 (GSE117358).
Il dataset contiene i dati da 48 topi con 2 tipi diversi di tumore:

* AB1 - mesotelioma (tumore della pleura)
* Renca — carcinoma renale

Per ogni modello tumorale hanno trattato i topi con immunoterapia (anticorpi anti-CTLA4 + anti-PD-L1) e poi li hanno divisi in base alla risposta:

Responders — il tumore ha risposto al trattamento
Non-responders — il tumore non ha risposto

L'RNAseq è fatto pre-trattamento :  l'idea è capire se esiste già una firma molecolare nel tumore che predice chi risponderà e chi no, prima ancora di iniziare la terapia.
In numeri:

- 24 campioni AB1 → 12 responders + 12 non-responders
- 24 campioni Renca → 12 responders + 12 non-responders

48 campioni totali

In [6]:
### Richiamiamo le librerie principali e installiamo pydeseq2 e GEOparse
import os
import pickle as pkl
import numpy as np

!pip install pydeseq2
!pip install GEOparse

In [7]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

In [16]:
# Scarichiamo ora i dati
import GEOparse
gse = GEOparse.get_GEO("GSE117358", destdir="./data")

27-May-2026 13:13:22 DEBUG utils - Directory ./data already exists. Skipping.
DEBUG:GEOparse:Directory ./data already exists. Skipping.
27-May-2026 13:13:22 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE117nnn/GSE117358/soft/GSE117358_family.soft.gz to ./data/GSE117358_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE117nnn/GSE117358/soft/GSE117358_family.soft.gz to ./data/GSE117358_family.soft.gz
100%|██████████| 6.14k/6.14k [00:00<00:00, 71.8kB/s]
27-May-2026 13:13:23 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
27-May-2026 13:13:23 DEBUG downloader - Moving /tmp/tmpc9h33rxt to /content/data/GSE117358_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpc9h33rxt to /content/data/GSE117358_family.soft.gz
27-May-2026 13:13:23 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE117nnn/GSE117358/soft/GSE117358_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.n

In [13]:
# Verichifichiamo il contenuto della cartella
! ls data


GSE52778_family.soft.gz


Leggiamo dall'oggetto GSE il contenuto dei dati.

In [17]:

print("GSM: GSE117358")
for gsm_name, gsm in gse.gsms.items():
    print("Name: ", gsm_name)
    print("Metadata:",)
    for key, value in gsm.metadata.items():
        print(" - %s : %s" % (key, ", ".join(value)))
    print ("Table data:",)
    print (gsm.table.head())
    break

GSM: GSE117358
Name:  GSM3291713
Metadata:
 - title : AB1 Responder rep1 [AB01]
 - geo_accession : GSM3291713
 - status : Public on Jul 18 2019
 - submission_date : Jul 19 2018
 - last_update_date : Nov 17 2020
 - type : SRA
 - channel_count : 1
 - source_name_ch1 : AB1_Responder
 - organism_ch1 : Mus musculus
 - taxid_ch1 : 10090
 - characteristics_ch1 : host strain: balb/c, host mouse age: 8-9 weeks, host mouse gender: female, tumor cells: AB1, tumor type: mesothelioma, treatment: anti-CTLA4 plus anti-PDL-1, timepoint: day 0 - pretreatment, response to immunotherapy: Responder, rna 260/280: 2.04, rna 260/230: 2.03
 - growth_protocol_ch1 : AB1 or Renca tumour cells were subcutaneously injected into Balb/c mice, and whole tumour surgically removed 8 or 10 days after (respectively)"
 - molecule_ch1 : total RNA
 - extract_protocol_ch1 : Frozen tumours were dissociated in Trizol using a TissueRuptor, total RNA was extracted using chloroform and purified on Rneasy MiniElute columns., RNA l

Leggiamo dove sono salvati i file con le conte.

In [18]:
print(gse.metadata['supplementary_file'])

['ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE117nnn/GSE117358/suppl/GSE117358_genecounts.csv.gz']


In [ ]:
import pandas as pd
url = gse.metadata['supplementary_file'][0]


In [25]:
counts = pd.read_csv(url, compression="gzip")
print(counts.shape)
print(counts.head())

(39637, 51)
   AB01   AB02  AB09   AB10  AB17  AB18  AB25   AB26  AB33   AB34  ...  RZ66  \
0  5344   6003  5392   5452  4652  4620  4510   5077  4313   6617  ...  3032   
1     0      0     0      0     0     0     0      0     0      0  ...     0   
2   812    967   841    882   658   434   686    867   568   1247  ...   766   
3  8346  14646  6320  16274  5830  5422  4158  13197  4356  13387  ...  1288   
4    88    142    83    128    70    75    51    107    45    143  ...    13   

   RZ67  RZ68  RZ69  RZ70  RZ71  RZ72           EnsemblID  Symbol  \
0  3635  3756  3248  3257  3548  3442  ENSMUSG00000000001   Gnai3   
1     0     0     0     0     0     0  ENSMUSG00000000003    Pbsn   
2   732   830   830   726   716   767  ENSMUSG00000000028   Cdc45   
3  2907  1896   515  1807  2643  1693  ENSMUSG00000000031     H19   
4    30    27    20    22    19    16  ENSMUSG00000000037   Scml2   

                                         Description  
0  guanine nucleotide binding protein

In [26]:
print(counts.columns)


# Verifichiamo che ci siano tutti i campioni nella matrice
print(len(counts.columns[:-3]))
counts.columns[:-3]

Index(['AB01', 'AB02', 'AB09', 'AB10', 'AB17', 'AB18', 'AB25', 'AB26', 'AB33',
       'AB34', 'AB41', 'AB42', 'AB49', 'AB50', 'AB57', 'AB58', 'AB65', 'AB66',
       'AB67', 'AB68', 'AB69', 'AB70', 'AB71', 'AB72', 'RZ01', 'RZ02', 'RZ09',
       'RZ10', 'RZ17', 'RZ18', 'RZ25', 'RZ26', 'RZ33', 'RZ34', 'RZ41', 'RZ42',
       'RZ49', 'RZ50', 'RZ57', 'RZ58', 'RZ65', 'RZ66', 'RZ67', 'RZ68', 'RZ69',
       'RZ70', 'RZ71', 'RZ72', 'EnsemblID', 'Symbol', 'Description'],
      dtype='object')
48


Index(['AB01', 'AB02', 'AB09', 'AB10', 'AB17', 'AB18', 'AB25', 'AB26', 'AB33',
       'AB34', 'AB41', 'AB42', 'AB49', 'AB50', 'AB57', 'AB58', 'AB65', 'AB66',
       'AB67', 'AB68', 'AB69', 'AB70', 'AB71', 'AB72', 'RZ01', 'RZ02', 'RZ09',
       'RZ10', 'RZ17', 'RZ18', 'RZ25', 'RZ26', 'RZ33', 'RZ34', 'RZ41', 'RZ42',
       'RZ49', 'RZ50', 'RZ57', 'RZ58', 'RZ65', 'RZ66', 'RZ67', 'RZ68', 'RZ69',
       'RZ70', 'RZ71', 'RZ72'],
      dtype='object')

In [31]:
samples=counts.columns[:-3].to_list()
AB1 = [sample for sample in samples if "AB" in sample]
RZ = [sample for sample in samples if "RZ" in sample]

print(f"Gli AB1 sono {len(AB1)}")
print(f"Gli RZ sono {len(RZ)}")


Gli AB1 sono 24
Gli RZ sono 24


 Modifichiamo la tabella delle count in modo da avere solo i samples e i nomi dei geni
 come indice della matrice. Inoltre pyDeseq2 vuole una matrice samples x geni, con i
 samples sulle righe e i geni come colonne (a differenza di Deseq2 in R). Quest'ultimo passaggio lo faremo subito prima di creare l'oggetto **DeseqDataSet**, prima faremo un po' di pulizia della matrice per rimuovere alcuni geni.



In [64]:
clean_counts=counts.iloc[:,0:-3]
clean_counts.set_index(counts["Symbol"])


,AB01,AB02,AB09,AB10,AB17,AB18,AB25,AB26,AB33,AB34,...,RZ57,RZ58,RZ65,RZ66,RZ67,RZ68,RZ69,RZ70,RZ71,RZ72
Symbol,,,,,,,,,,,,,,,,,,,,,
Gnai3,5344,6003,5392,5452,4652,4620,4510,5077,4313,6617,...,3398,3691,3133,3032,3635,3756,3248,3257,3548,3442
Pbsn,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cdc45,812,967,841,882,658,434,686,867,568,1247,...,569,844,701,766,732,830,830,726,716,767
H19,8346,14646,6320,16274,5830,5422,4158,13197,4356,13387,...,4234,1784,1331,1288,2907,1896,515,1807,2643,1693
Scml2,88,142,83,128,70,75,51,107,45,143,...,18,22,22,13,30,27,20,22,19,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RP24-148D3.3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
RP23-48A24.3,68,96,76,87,68,70,51,113,79,124,...,89,75,82,78,922,76,68,55,92,75
RP23-255F14.9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


 Dobbiamo ora creare, visto che non è fornita direttamente, la tabella con i metadati per ogni campione.
 Vediamo cosa abbiamo per ogni campione all'interno dell'oggetto GSE

In [33]:
import pandas as pd

rows = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata['title'][0]
    characteristics = gsm.metadata['characteristics_ch1']
    rows.append({
        "sample": gsm_name,
        "title": title,
        "characteristics": str(characteristics)
    })

meta_df_temp = pd.DataFrame(rows)
print(meta_df_temp.head(10).to_string(index=False))

    sample                         title                                                                                                                                                                                                                                                                                     characteristics
GSM3291713     AB1 Responder rep1 [AB01]     ['host strain: balb/c', 'host mouse age: 8-9 weeks', 'host mouse gender: female', 'tumor cells: AB1', 'tumor type: mesothelioma', 'treatment: anti-CTLA4 plus anti-PDL-1', 'timepoint: day 0 - pretreatment', 'response to immunotherapy: Responder', 'rna 260/280: 2.04', 'rna 260/230: 2.03']
GSM3291714 AB1 Non-responder rep1 [AB02] ['host strain: balb/c', 'host mouse age: 8-9 weeks', 'host mouse gender: female', 'tumor cells: AB1', 'tumor type: mesothelioma', 'treatment: anti-CTLA4 plus anti-PDL-1', 'timepoint: day 0 - pretreatment', 'response to immunotherapy: Non-responder', 'rna 260/280: 2.05', 'rna 260/230: 2.21']
G

Cerchiamo di sintetizzare le informazioni contenute per ottenere un dataframe con 3 colonne:
 * SampleID
 * Tumor_Type
 * Treatment_Response

In [37]:
meta_df_temp["SampleID"]=meta_df_temp["title"].apply(lambda x: x.split("[")[1].split("]")[0])




In [48]:
import ast
meta_df_temp["Tumor_Type"]=meta_df_temp["characteristics"].apply(lambda x: ast.literal_eval(x)[3].replace("tumor cells: ",""))
meta_df_temp["Treatment_Response"]=meta_df_temp["characteristics"].apply(lambda x: ast.literal_eval(x)[7].replace("response to immunotherapy: ",""))



In [50]:
metadata=meta_df_temp[["SampleID","Tumor_Type","Treatment_Response"]]

metadata.head()

,SampleID,Tumor_Type,Treatment_Response
0,AB01,AB1,Responder
1,AB02,AB1,Non-responder
2,AB09,AB1,Responder
3,AB10,AB1,Non-responder
4,AB17,AB1,Responder


In [52]:
pd.crosstab(metadata["Tumor_Type"],metadata["Treatment_Response"])

Treatment_Response,Non-responder,Responder
Tumor_Type,,
AB1,12,12
Renca,12,12


Abbiamo ora i due componenti fondamentali per iniziare l'analisi:
la matrice con le count e i metadata.